### Incorporation of an external AI into Jupyter Notebook for Python
A free API of Gemini is used in this notebook. 

1. Create a text file named `.env` in the same folder as the notebook and enter the API key (never make this file public).
   GEMINI_API_KEY=A**** (your AIP key)
2. If you are managing the project with GitHub, add '.env' to your .gitignore file to prevent .env file from being uploaded to the public repository.

In [ ]:
# Integrating AI
!pip install google-genai

In [ ]:
# AIP environment
!pip install python-dotenv

In [ ]:
# List of available AI models
from dotenv import load_dotenv
from google import genai

load_dotenv()
client = genai.Client()

# List of available AI models
for model in client.models.list():
    print(model.name)

### Exercise of finding reduced echelon forms

Execute the next cell to create sliders. Choose the size $(m,n)$ of a matrix.

In [ ]:
import ipywidgets as widgets
from IPython.display import display

slider1 = widgets.IntSlider(value=3, min=2, max=6, description='m')
slider2 = widgets.IntSlider(value=4, min=2, max=6, description='n')

display(slider1, slider2)

Execute the next cell to create an $m{\times}n$ matrix, and find its reduced echelon form. AI also shows the solution.

In [ ]:
import json
import re
from google import genai
from IPython.display import Markdown, display
import sympy as sp

# 1. getting structured data by json
prompt = f"""
Create a {m} by {n} integer matrix A with non-trivial row operations, and find its reduced row echelon form (RREF) step by step.

Requirements for steps:
- For every elementary row operation, state the operation (e.g., $R_2 \\to R_2 - 2R_1$).
- Immediately after each operation, display the **entire intermediate matrix** using LaTeX bmatrix format ($$\\begin{{bmatrix}} ... \\end{{bmatrix}}$$), not just the modified row.
- Conclude with the final reduced row echelon form matrix.

Output format (JSON):
{{
  "matrix_A": [[...]],
  "ai_rref": [[...]],
  "steps_explanation": "Markdown string containing each operation and its full intermediate LaTeX matrix."
}}
"""

response = client.models.generate_content(
    model="gemini-3.5-flash-lite",
    contents=prompt,
    config={"response_mime_type": "application/json"},  # structured output
)

data = json.loads(response.text)

# 2. computing RREF by Python (SymPy) 
A_list = data["matrix_A"]
A_sym = sp.Matrix(A_list)
true_rref, pivots = A_sym.rref()  

# converting AI output and compare it with sympy output
ai_rref_sym = sp.Matrix(data["ai_rref"])
is_correct = true_rref.equals(ai_rref_sym)

# 3. problem, solution, and verification results 
display(
    Markdown(
        f"### An original matrix $A$\n$${sp.latex(A_sym)}$$\n\n"
        f"### The answer generated by AI\n{data['steps_explanation']}\n\n"
        f"---"
    )
)

if is_correct:
    display(
        Markdown(
            " **Verification results (Python / SymPy):** The answer generated by AI is correct."
        )
    )
else:
    display(
        Markdown(
            " **Verification results (Python / SymPy):** The answer generated by AI is incorrect\n\n"
            f"**Correct RREF:**\n$${sp.latex(true_rref)}$$"
        )
    )

### Exercise of computing determinants

Execute the next cell to create a slider. Choose the size $n$ of a square matrix. 

In [ ]:
import ipywidgets as widgets
from IPython.display import display

slider3 = widgets.IntSlider(value=4, min=2, max=6, description='n')

display(slider3)

Execute the next cell to create a square matrix, and compute its reduced determinant. AI also shows the solution.

In [ ]:
# Exercise of determinants
from IPython.display import Markdown, display
from dotenv import load_dotenv
from google import genai

load_dotenv()
client = genai.Client()

n = slider3.value

prompt = f"""
Create a {n} by {n} integer matrix A.
Include:
1. The original matrix A.
2. Compute the determinant of A using at least one method that involves elementary row and columin operations. Show the detail of the process.
3. If the determinant is not zero, obtain its inverse. Show the detail of the process.
"""

response = client.models.generate_content(
    model="gemini-3.5-flash-lite",
    contents=prompt,
)

display(Markdown(response.text))